In [1]:
import json
import numpy as np
import pandas as pd

with open("../models/feature_mapping.json") as f:
    mapping = json.load(f)      # {model feature name: tool column name}

print("Mapped features:", len(mapping))

Mapped features: 66


In [2]:
flows = pd.read_csv("../data/my_flows.csv")

dur_tool = mapping["Flow Duration"]
print("Longest flow in YOUR capture:", flows[dur_tool].max())
print("Median flow in YOUR capture:", flows[dur_tool].median())

train_peek = pd.read_csv("../data/Wednesday-workingHours.pcap_ISCX.csv", nrows=50000)
train_peek.columns = train_peek.columns.str.strip()
print("Longest flow in the DATASET (first 50k rows):", train_peek["Flow Duration"].max())

Longest flow in YOUR capture: 15026850.0
Median flow in YOUR capture: 231234.0
Longest flow in the DATASET (first 50k rows): 119999979


In [3]:
TIME_SCALE = 1_000_000   # change to 1 if your units already match (see Cell 2)

model_features = list(mapping.keys())

df = pd.read_csv("../data/Wednesday-workingHours.pcap_ISCX.csv")
df.columns = df.columns.str.strip()

df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.drop_duplicates()

y = (df["Label"] != "BENIGN").astype(int)
X = df[model_features]

print("Rows:", len(X), "| Features:", X.shape[1])
print("Share of attacks: {:.1%}".format(y.mean()))
print(df["Label"].value_counts())

Rows: 610492 | Features: 66
Share of attacks: 31.7%
Label
BENIGN              416736
DoS Hulk            172846
DoS GoldenEye        10286
DoS slowloris         5385
DoS Slowhttptest      5228
Heartbleed              11
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import time

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Trained in {:.0f} seconds".format(time.time() - start))

Trained in 34 seconds


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

pred = rf.predict(X_test)
print(classification_report(y_test, pred, target_names=["normal", "attack"], digits=4))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

      normal     0.9991    0.9990    0.9991     83348
      attack     0.9979    0.9982    0.9980     38751

    accuracy                         0.9988    122099
   macro avg     0.9985    0.9986    0.9986    122099
weighted avg     0.9988    0.9988    0.9988    122099

[[83267    81]
 [   71 38680]]


In [6]:
import joblib, os

os.makedirs("../models", exist_ok=True)
joblib.dump(
    {"model": rf, "features": model_features, "mapping": mapping, "time_scale": TIME_SCALE},
    "../models/nids_rf.joblib",
)
print("Saved. Size (MB):", round(os.path.getsize("../models/nids_rf.joblib") / 1e6, 1))

Saved. Size (MB): 16.8


In [7]:
TIME_FEATURES = [f for f in model_features
                 if any(k in f for k in ("Duration", "IAT", "Active", "Idle"))]
print("Time-based features rescaled by", TIME_SCALE, ":", len(TIME_FEATURES))

flows = pd.read_csv("../data/my_flows.csv")

# Build the model's input table: one column per model feature, in the same order
Xnew = pd.DataFrame({feat: flows[tool_col] for feat, tool_col in mapping.items()})
Xnew = Xnew[model_features]

for feat in TIME_FEATURES:
    Xnew[feat] = Xnew[feat] * TIME_SCALE

Xnew = Xnew.replace([np.inf, -np.inf], np.nan).fillna(0)
print(Xnew.shape)

Time-based features rescaled by 1000000 : 23
(71, 66)


In [8]:
flows["prediction"] = rf.predict(Xnew)
flows["attack_probability"] = rf.predict_proba(Xnew)[:, 1]

print(flows["prediction"].value_counts().rename({0: "normal", 1: "attack"}))

prediction
normal    71
Name: count, dtype: int64


In [9]:
cols = ["src_ip", "src_port", "dst_ip", "dst_port", "protocol", "attack_probability"]
flows[flows["prediction"] == 1][cols].sort_values("attack_probability", ascending=False).head(20)

,src_ip,src_port,dst_ip,dst_port,protocol,attack_probability
